# RNAbpFlow Pilot10 — structural accuracy evaluation (CPU, Drive-backed)

This notebook evaluates **RNAbpFlow only** (single `sample_0.pdb` from each of 10 frozen targets). It uses the **same native-mapped, native-normalized US-align v20260527** protocol as the preceding DRfold evaluation. GPU is **not** required. Native experimental structures are read **only for evaluation**, never inference. RNAbpFlow predictions were conditioned on frozen **predicted IPKnot secondary structures**, not native base pairs.

**Safety:** 10/10 complete RNA predictions and their hashes are verified before scoring; no overwrites of saved evaluation results; a `FINALIZED.json` hash inventory is created and copied to Drive only upon successful 10-target evaluation. No GitHub writes.

## 1. Mount Drive and inspect inputs (no upload)

In [ ]:
from pathlib import Path
from google.colab import drive
import csv, hashlib, json, shutil, zipfile, os, re, subprocess, math
from datetime import datetime, timezone
from statistics import mean

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/RNA3D_Benchmark')
PRED_ROOT=ROOT/'RNAbpFlow/runs/rnabpflow_pilot10_ipknot_singlemap_v1'
NATIVE_ZIP=ROOT/'DRfold/Evaluations/RNA3D_DRfold_Pilot10_Eval_Inputs_20261119.zip'
EVAL_ROOT=ROOT/'RNAbpFlow/Evaluations'
TARGETS=['8ubd_H','9fia_bJ','9cfn_B','9m31_B','9g4p_B','9lq7_K','8z8u_B','9g6k_lB','9cf2_W','8wf8_B']
MODEL_SHA='e8b1c077f7951d4d6ebe00c851d22c37777dfc9f'
CKPT_SHA='bb598a736193734541f63c59dae75d45db8c04f01a8414a758677367c4b1e213'
USALIGN_SHA='177cc8a2bbd3e2a6e9c5faaaa4ff5dfa1e6048f7'
RUN_NAME='rnabpflow_pilot10_ipknot_usalign20260527_20261119_v1'
assert PRED_ROOT.is_dir(),f'Missing RNAbpFlow prediction Drive directory: {PRED_ROOT}'
assert NATIVE_ZIP.is_file(),f'Missing existing native reference bundle: {NATIVE_ZIP}'
assert NATIVE_ZIP.stat().st_size==395773,'Native reference bundle size has changed'
with zipfile.ZipFile(NATIVE_ZIP) as z:
    assert z.testzip() is None, 'Corrupt reference ZIP'
    for t in TARGETS:
        assert f'native/{t}_native_eval.pdb' in z.namelist()
for t in TARGETS:
    d=PRED_ROOT/t
    assert (d/'complete.json').is_file() and (d/f'{t}_sample_0.pdb').is_file(),f'{t}: missing prediction/complete marker'
print('INPUT VERIFICATION PASSED: ten prediction folders plus ten native references available')
print('Prediction root:',PRED_ROOT)
print('Native bundle:',NATIVE_ZIP)


## 2. Build pinned US-align (CPU only)

In [ ]:
SRC=Path('/content/USalign_rnabpflow_pilot10_20260527')
TAG='20260527'
if not SRC.exists():
    subprocess.run(['git','clone','--depth','1','--branch',TAG,'https://github.com/pylelab/USalign.git',str(SRC)],check=True)
actual=subprocess.check_output(['git','-C',str(SRC),'rev-parse','HEAD'],text=True).strip()
assert actual==USALIGN_SHA,f'Unexpected USalign commit {actual}; expected {USALIGN_SHA}'
USALIGN=SRC/'USalign'
if not USALIGN.is_file(): subprocess.run(['make'],cwd=SRC,check=True)
assert USALIGN.is_file() and os.access(USALIGN,os.X_OK)
print('USalign pinned:',actual)


## 3. Validated mapping and scoring functions

In [ ]:
def digest(data): return hashlib.sha256(data).hexdigest()

def residues(data):
    records=[];seen=set()
    for l in data.decode(errors='replace').splitlines():
        if not l.startswith('ATOM  '):continue
        key=(l[21:22],l[22:26],l[26:27],l[17:20].strip())
        if key not in seen:
            seen.add(key)
            records.append({'key':key,'nt':l[17:20].strip(),'position':int(l[22:26])})
    return records

def seq_from_fasta(data):
    return ''.join(l.strip() for l in data.decode().splitlines() if l.strip() and not l.startswith('>')).upper()

def map_to_native_positions(pred,native_positions):
    r=residues(pred)
    order={res['key']:i+1 for i,res in enumerate(r)}
    out=[];serial=1
    for l in pred.decode(errors='replace').splitlines():
        if not l.startswith(('ATOM  ','HETATM')):continue
        if l[16:17] not in (' ','A','1'):continue
        key=(l[21:22],l[22:26],l[26:27],l[17:20].strip())
        pos=order.get(key)
        if pos not in native_positions:continue
        out.append(l[:6]+f'{serial:5d}'+l[11:22]+f'{pos:4d}'+l[26:])
        serial+=1
    assert out, 'No mapped atoms'
    return ('\n'.join(out+['TER','END'])+'\n').encode()

def parse_usalign(output,n_native):
    m=re.search(r'Aligned length\s*=\s*(\d+)\s*,\s*RMSD\s*=\s*([\d.eE+-]+)',output,re.I)
    if not m:raise ValueError('Missing aligned length and RMSD in USalign output')
    aligned,rmsd=int(m.group(1)),float(m.group(2))
    matches=re.findall(r'TM-score\s*=\s*([\d.eE+-]+)\s*\(normalized by length of Structure_2\b[^)]*\)',output,re.I)
    if not matches:raise ValueError('Missing Structure_2/native-normalized TM-score')
    tm=float(matches[-1])
    assert 0<aligned<=n_native and math.isfinite(rmsd) and rmsd>=0 and 0<=tm<=1.00001
    return {'tm_score':tm,'rmsd_angstrom':rmsd,'alignment_residues':aligned,'alignment_coverage':aligned/n_native}

assert parse_usalign('Aligned length= 13, RMSD= 11.52, Seq_ID=n_identical/n_aligned= 1.000\nTM-score= 0.24331 (normalized by length of Structure_1: L=13, d0=0.40)\nTM-score= 0.24331 (normalized by length of Structure_2: L=13, d0=0.40)',13)['tm_score']==0.24331
print('Metric parser self-test PASS')


## 4. Calculate RNAbpFlow structural scores for all ten targets

In [ ]:
LOCAL=Path('/content')/RUN_NAME
assert not LOCAL.exists(),f'Local evaluation exists; rerunning would overwrite it: {LOCAL}'
assert not (EVAL_ROOT/RUN_NAME).exists(),f'Saved evaluation already exists on Drive; do not overwrite: {EVAL_ROOT/RUN_NAME}'
LOCAL.mkdir()
(LOCAL/'logs').mkdir();(LOCAL/'mapped').mkdir()
rows=[]
with zipfile.ZipFile(NATIVE_ZIP) as archive:
    for t in TARGETS:
        folder=PRED_ROOT/t
        meta=json.loads((folder/'complete.json').read_text())
        assert meta.get('status')=='PASS' and meta.get('model')=='RNAbpFlow' and meta.get('target_id')==t
        assert meta.get('model_commit')==MODEL_SHA and meta.get('checkpoint_sha256')==CKPT_SHA
        assert meta.get('input_mode')=='fasta_predicted_ipknot_single_binary_bp_map'
        assert meta.get('native_used_for_inference') is False and meta.get('native_bp_used_for_inference') is False
        assert meta.get('exit_code')==0 and meta.get('n_samples')==1
        samples=meta.get('prediction_files',[])
        assert len(samples)==1 and samples[0]['filename']==f'{t}_sample_0.pdb'
        seq_bytes=(folder/f'{t}.fasta').read_bytes()
        pred_bytes=(folder/f'{t}_sample_0.pdb').read_bytes()
        map_bytes=(folder/'map1.npy').read_bytes()
        assert digest(seq_bytes)==meta['fastasha256'] and digest(pred_bytes)==samples[0]['sha256']
        assert digest(map_bytes)==meta['map1_sha256']
        seq=seq_from_fasta(seq_bytes)
        pred_res=residues(pred_bytes)
        assert set(seq)<=set('ACGU') and len(seq)==meta['input_length']==len(pred_res)==samples[0]['residues'],t
        assert ''.join(r['nt'] for r in pred_res)==seq,f'Prediction residue sequence mismatch: {t}'
        nat_bytes=archive.read(f'native/{t}_native_eval.pdb')
        native_res=residues(nat_bytes)
        positions=[r['position'] for r in native_res]
        assert len(set(positions))==len(positions)>0 and all(1<=r['position']<=len(seq) and seq[r['position']-1]==r['nt'] for r in native_res),t
        mapped=map_to_native_positions(pred_bytes,set(positions))
        pred_mapped=LOCAL/'mapped'/f'{t}_rnabpflow_mapped.pdb'
        native_mapped=LOCAL/'mapped'/f'{t}_native_eval.pdb'
        pred_mapped.write_bytes(mapped)
        native_mapped.write_bytes(nat_bytes)
        p=subprocess.run([str(USALIGN),str(pred_mapped),str(native_mapped),'-mol','RNA','-TMscore','1'],capture_output=True,text=True)
        output=p.stdout + ('\n--- STDERR ---\n'+p.stderr if p.stderr else '')
        (LOCAL/'logs'/f'{t}_usalign.txt').write_text(output)
        assert p.returncode==0,f'{t}: USalign failed code {p.returncode} — {output[-1200:]}'
        score=parse_usalign(p.stdout,len(native_res))
        rows.append(dict(target_id=t,model='RNAbpFlow',input_length=len(seq),native_residues=len(native_res),native_fraction=len(native_res)/len(seq),bp_input='predicted_IPKnot_single_map',sample_id=0,prediction_sha256=digest(pred_bytes),mapped_sha256=digest(mapped),runtime_seconds=meta['runtime_seconds'],**score,status='scored'))
        print(f"{t:9s} TM={score['tm_score']:.5f} RMSD={score['rmsd_angstrom']:.2f} Å aligned={score['alignment_residues']}/{len(native_res)}",flush=True)
assert len(rows)==len(TARGETS)==10
with (LOCAL/'rnabpflow_scores.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=rows[0].keys(),lineterminator='\n');writer.writeheader();writer.writerows(rows)
summary={'status':'complete','target_count':10,'model':'RNAbpFlow','model_commit':MODEL_SHA,'checkpoint_sha256':CKPT_SHA,'conditioning':'predicted IPKnot single binary map, one sampled structure per target, sample_id=0','usalign_version':'20260527','usalign_source_commit':USALIGN_SHA,'method':'USalign mapped_prediction native_eval -mol RNA -TMscore 1 (Structure_2 native normalization)','mean_tm':mean(r['tm_score'] for r in rows),'mean_rmsd_angstrom':mean(r['rmsd_angstrom'] for r in rows),'limitations':['exploratory 10-target panel','model receives predicted secondary-structure conditioning','one frozen sample per target (not native-selected best-of-N)','incomplete native experimental coordinate coverage for some targets','template scarcity and checkpoint training overlap not established']}
(LOCAL/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
filehashes={str(p.relative_to(LOCAL)):digest(p.read_bytes()) for p in sorted(LOCAL.rglob('*')) if p.is_file()}
(LOCAL/'FINALIZED.json').write_text(json.dumps({'status':'complete','targets':10,'created_utc':datetime.now(timezone.utc).isoformat(),'files':filehashes},indent=2)+'\n')
assert all(digest((LOCAL/k).read_bytes())==v for k,v in filehashes.items())
print('FINAL VERIFICATION PASS: 10/10 RNAbpFlow targets scored')
print(f"RNAbpFlow mean TM={summary['mean_tm']:.5f}; RMSD={summary['mean_rmsd_angstrom']:.2f} Å")


## 5. Save immutable results and backup ZIP to Google Drive

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import display, FileLink
assert (LOCAL/'FINALIZED.json').is_file(), 'Incomplete evaluation: refusing to save'
EVAL_ROOT.mkdir(parents=True,exist_ok=True)
DEST=EVAL_ROOT/RUN_NAME
BACKUP=Path('/content/RNA3D_RNAbpFlow_Pilot10_Evaluation_Results.zip')
REMOTE_BACKUP=EVAL_ROOT/BACKUP.name
assert not BACKUP.exists() and not DEST.exists() and not REMOTE_BACKUP.exists(), 'Existing destination; never overwrite'
with ZipFile(BACKUP,'x',compression=ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(LOCAL.rglob('*')):
        if p.is_file():z.write(p,arcname=RUN_NAME+'/'+str(p.relative_to(LOCAL)))
with ZipFile(BACKUP) as z:assert z.testzip() is None
shutil.copytree(LOCAL,DEST)
shutil.copy2(BACKUP,REMOTE_BACKUP)
for p in LOCAL.rglob('*'):
    if p.is_file():
        other=DEST/p.relative_to(LOCAL)
        assert other.is_file() and digest(other.read_bytes())==digest(p.read_bytes()),f'Drive mismatch: {p}'
assert digest(BACKUP.read_bytes())==digest(REMOTE_BACKUP.read_bytes())
print('✅ FINAL DRIVE VERIFICATION PASS: 10/10 RNAbpFlow scores saved')
print('Drive results:',DEST)
print('Drive archive:',REMOTE_BACKUP)
display(FileLink(str(BACKUP)))


## 6. Read-only re-verification (safe to rerun)

In [ ]:
SAVED=EVAL_ROOT/RUN_NAME
marker=SAVED/'FINALIZED.json'
assert marker.is_file(),f'Missing finalized run: {marker}'
record=json.loads(marker.read_text())
assert record['status']=='complete' and record['targets']==10
for path,checksum in record['files'].items():
    p=SAVED/path
    assert p.is_file() and digest(p.read_bytes())==checksum, f'Altered or missing saved artifact: {path}'
with (SAVED/'rnabpflow_scores.csv').open() as f:rr=list(csv.DictReader(f))
assert len(rr)==10 and set(r['target_id'] for r in rr)==set(TARGETS) and all(r['status']=='scored' for r in rr)
print('FINAL DRIVE VERIFICATION PASS: 10/10 RNAbpFlow scores intact')
print('Summary:',json.loads((SAVED/'summary.json').read_text()))
